In [ ]:
import ollama

In [ ]:
example = """<think>I see I have access to these set of tools: ['quote_get_option_fundamentals']. I have to validate which tools to call and how many tool calls are required. Let's solve it step by step.

I will begin by activating the option fundamentals tool within the fastAPI function, 'quote_get_option_fundamentals' from the function. I also need to provide the currency code for the option fundamentals. 

Since I don't have these parameters, I will proceed with the other tool names and call methods with those required parameters. 

I will use the 'quote_get_option_fundamentals' function from the QuickAPI and select either 'V' or 'MA'. 

Please note that the relevant tools are 'quote_get_option_fundamentals' and 'quote_get_options_fundamentals'. 

Let's start by connecting them to Mastercard with the correct currency code.</think>
<tool_call>[{'name': 'quote_get_option_fundamentals', 'arguments': {'symbol': 'V'}}, {'name': 'quote_get_option_fundamentals', 'arguments': {'symbol': 'MA'}}]</tool_call>"""


# example = """<think>For the given question, I have to call 'quote_get_option_fundamentals' with attribute symbol where the value would be 'V'.</think>
# <tool_call>[{'name': 'quote_get_option_fundamentals', 'arguments': {'symbol': 'V'}}]</tool_call>"""

In [ ]:
def thinking_validate(llm_gen):
    TOOL_VERIFY_PROMPT = """"You will be given a chain of thought of an LLM that tries to plan and make function/tool call. 
The step by step thinking/planning would be inside <think> </think> tags and the tool/function call would be inside <tool_call> </tool_call> tags. 
You have to validate if the step by step is reflecting the tool calls properly or not.


# Instructions:

- If the 'think' is not reflecting the actions made in inside 'tool_call': return False
- If the 'think' is not contextually idealizing the actions inside 'tool_call': return False
- If the 'think' highlighting different actions than taken inside 'tool_call': return False
- If the 'think' is indicating multiple steps but execute only one tool: return False

If all of the above instructions pass, return True


# Response Format:

Respond either True of False.
Example:
[True/False]
"""

    # def chat_with_ollama(example):
    messages = [
        # {"role": "system", "content": "You are a helpful AI assistant named SmolThink. First plan/reason/code/validate inside <think></think> tag and provide final answer to user query inside <answer></answer> tag."},
        {"role": "system", "content": TOOL_VERIFY_PROMPT},
        {"role": "user", "content": llm_gen}
    ]

    # First call: Initial chat with streaming enabled
    resp = ollama.chat(
        model="qwen3:0.6b",
        messages=messages,
        stream=False,
        think=False,
        options={
            # 'ctx': 512,
            'temperature': 0.,
            'num_predict': 2
        },   
    )

    return resp.message.content.strip().lower() == 'true'

print(thinking_validate(example))

In [ ]:
response_stream = ollama.chat(
        model="qwen3:0.6b",
        messages=[{"role": "user", "content": "Hi!"}],
        stream=False,
        think=False,
        options={
            'ctx': 512,
            'temperature': 0.,
            'num_predict': 12
        },
    )

print(response_stream.message['content'])

In [ ]:
from ddgs import DDGS

def web_search(query: str, max_results: int = 2) -> str:
    """
    Performs a DuckDuckGo web search and returns formatted markdown output.
    
    Args:
        query (str): Search query string.
        max_results (int): Number of results to return.
    
    Returns:
        str: Markdown formatted search results.
    """
    results = DDGS().text(query, max_results=max_results)
    
    output_lines = ["## Search Results"]
    
    for i, item in enumerate(results):
        title = item.get("title", "No Title")
        href = item.get("href", "")
        snippet = item.get("body", "")
        date = item.get("date", "Unknown date")
        
        output_lines.append(
            f"{i}. [{title}]({href})\n"
            f"Date published: {date}\n\n"
            f"{snippet}\n"
        )
    
    return "\n".join(output_lines)


# Example use:
if __name__ == "__main__":
    print(duck_search("Nansei Islands"))


In [3]:
import json

with open("/Users/ohi/Documents/GitHub/NanoAgent/weights/NanoAgent-135M-grpo-hilr/train_info.json", "r") as f:
    train_info = json.load(f)

# print("Model loaded", flush=True)
# return (
#     train_info["iter_step"],
#     train_info["losses"],
#     train_info["learning_rates"],
#     train_info["all_rewards"],
#     train_info["max_rewards"],
#     train_info["std_rewards"],
#     train_info["tool_call_complexity"],
#     train_info["total_prompt_tokens"],
# )

    # train_info["iter_step"],
last = 100
all_losses = train_info["losses"]#[-last:]
learning_rates = train_info["learning_rates"]#[-last:]
all_rewards = train_info["all_rewards"]#[-last:]
max_rewards = train_info["max_rewards"]
    # train_info["std_rewards"],
# tool_call_complexity = train_info["tool_call_complexity"]#[-last:]
    # train_info["total_prompt_tokens"]
# print()

In [4]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d
from scipy.interpolate import make_interp_spline
from scipy.ndimage import gaussian_filter1d

def mean_map(data, win=10):
    def _mean(x):
        while len(x) < win: x.append(0)
        return sum(x) / len(x)
    _data = []
    for i in range(len(data)):
        data_win = data[max(0, i-win+1):i+1]
        # print(i, len(data_win))
        _data.append(_mean(data_win))
    return _data



# plt.close()
fig, axes = plt.subplots(4, 1, figsize=(18, 12), dpi=600)
fig.suptitle(f"GRPO Iter: {iter}", fontsize=13)

# GRPO Loss
axes[0].plot(
    np.cumsum(all_losses) / (np.arange(len(all_losses)) + 1), color="tab:red"
)
# axes[0].plot(gaussian_filter1d(all_losses, sigma=2), color="tab:red", linewidth=2)
# axes[0].plot(mean_map(all_losses, 50), color="tab:red", alpha=0.2)

# axes[0].plot(
#     mean_map(all_losses), color="tab:red"
# )
# axes[0].plot(
#     all_losses, color="tab:red", alpha=0.1
# )
axes[0].set_title("Training Loss")
axes[0].grid(True)
# axes[0].set_ylim([-0.03, -0.05])

# all_rewards = [0 for _ in range(10)] + all_rewards

# Rewards
axes[1].plot(
    np.cumsum(all_rewards) / (np.arange(len(all_rewards)) + 1), color="tab:blue"
)
# axes[1].plot(
#     mean_map(all_rewards), color="tab:blue"
# )

# spl = make_interp_spline(all_rewards, np.arange(len(all_rewards)), k=3)
# power_smooth = spl(all_rewards)
# axes[1].plot(gaussian_filter1d(all_rewards, sigma=2.5), linewidth=2, color="tab:blue")
# axes[1].plot(mean_map(all_rewards, 50), alpha=1, color="tab:blue")
# axes[1].axhline(y=max(all_rewards), color='tab:blue', linestyle='--')
# axes[1].plot(
#     all_rewards, color="tab:blue", alpha=0.1
# )
# axes[1].plot(
#     np.cumsum(max_rewards) / (np.arange(len(max_rewards)) + 1), color="tab:orange"
# )
axes[1].set_title("All Reward (blue) | Max Reward (orange)")
axes[1].grid(True)
# axes[1].set_ylim([0, 0.2])


axes[2].plot(gaussian_filter1d(max_rewards, sigma=2.5), linewidth=2, color="tab:orange")
# axes[2].plot(max_rewards, alpha=0.6, color="tab:orange")
axes[2].plot(
    np.cumsum(max_rewards) / (np.arange(len(max_rewards)) + 1), color="tab:blue"
)
axes[2].grid(True)

# Tool call complexity
# axes[2].plot(tool_call_complexity, color="tab:purple")
# axes[2].set_title("Tool Defs")
# axes[2].grid(True)

# total_prompt_tokens
axes[3].plot(learning_rates, color="tab:orange")
axes[3].set_title("Learning Rate")
axes[3].grid(True)

# Remove [0, 1] default ticks on x-axis for bottom graph
plt.tight_layout(pad=1)
# if save_path:
#     plt.savefig(
#         f"{save_path}/plot.jpg", dpi=400, bbox_inches="tight", transparent=True
#     )
# if plot:
plt.show()

In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("nvidia/Nemotron-Cascade-RL-Instruction-Following")

/Users/ohi/Documents/GitHub/NanoAgent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 108938/108938 [00:00<00:00, 519363.16 examples/s]


In [5]:
ds['train']['instruction_id_list']

Column([['detectable_format:title', 'length_constraints:number_words'], ['detectable_format:number_bullet_lists', 'change_case:english_lowercase'], ['length_constraints:number_sentences', 'detectable_format:number_highlighted_sections'], ['length_constraints:number_words', 'startend:end_checker'], ['detectable_format:number_bullet_lists', 'startend:end_checker']])

In [4]:
ds['train']

Dataset({
    features: ['prompt', 'instruction_id_list', 'kwargs', 'index'],
    num_rows: 108938
})

In [3]:
!pip install docker

In [2]:
import docker
import os
from typing import Optional


class DockerSandbox:
    def __init__(self):
        self.client = docker.from_env()
        self.container = None

    def create_container(self):
        try:
            image, build_logs = self.client.images.build(
                path=".",
                tag="agent-sandbox",
                rm=True,
                forcerm=True,
                buildargs={},
                # decode=True
            )
        except docker.errors.BuildError as e:
            print("Build error logs:")
            for log in e.build_log:
                if 'stream' in log:
                    print(log['stream'].strip())
            raise

        # Create container with security constraints and proper logging
        self.container = self.client.containers.run(
            "agent-sandbox",
            name="python-sandbox",
            command="tail -f /dev/null",  # Keep container running
            detach=True,
            tty=True,
            mem_limit="64m",
            cpu_quota=50000,
            pids_limit=100,
            security_opt=["no-new-privileges"],
            cap_drop=["ALL"],
            environment={
                "HF_TOKEN": os.getenv("HF_TOKEN")
            },
        )

    def run_code(self, code: str) -> Optional[str]:
        if not self.container:
            self.create_container()

        # Execute code in container
        exec_result = self.container.exec_run(
            cmd=["python", "-c", code],
            user="nobody"
        )

        # Collect all output
        return exec_result.output.decode() if exec_result.output else None


    def cleanup(self):
        if self.container:
            try:
                self.container.stop()
            except docker.errors.NotFound:
                # Container already removed, this is expected
                pass
            except Exception as e:
                print(f"Error during cleanup: {e}")
            finally:
                self.container = None  # Clear the reference

# Example usage:
sandbox = DockerSandbox()

# try:
if True:
    # Define your agent code
    agent_code = """
print('Hello world')
"""

    for i in range(10):
        # Run the code in the sandbox
        output = sandbox.run_code(agent_code)
        print(output)

# finally:
#     sandbox.cleanup()

Hello world

Hello world

Hello world

Hello world

Hello world

Hello world

Hello world

Hello world

Hello world

Hello world



In [6]:
sandbox.cleanup()

In [1]:
!pip install reasoning-gym

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 3.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 3.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 3.5 MB/s  0:00:00 eta 0:00:01
  Created wheel for cellpylib: filename=cellpylib-2.4.0-py3-none-any.whl size=38009 sha256=64a9b5ed791b63d58f64f97da92f72f9d09a6f46bc6c010df005367f2852a102
  Stored in directory: /Users/ohi/Library/Caches/pip/wheels/c4/eb/d4/86fddbd7c664c7a5cad193961c33db3e889d92e40d9f5aa9ab
  Created wheel for pycosat: filename=pycosat-0.6.6-cp313-cp

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial

# generate a number_sorting dataset

def number_sorting(tokenizer, size):
    dataset = reasoning_gym.create_dataset(
        name="number_sorting",   # task name
        min_numbers = 3,
        max_numbers = 10,
        min_decimals = 0,
        max_decimals = 1,
        min_value = -20,
        max_value = 20,
        seed = 42,
        size = size,
        # num_fewshot=1,
        # fewshot_as_multiturn=1
    )
    score_fn = get_score_answer_fn("number_sorting")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        last_line = last_line[last_line.find('['):]
        return score_fn(last_line, entry)

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = number_sorting(tokenizer=tokenizer, size=10)

print(ds[0]['prompt'])
print('larelappa\nFinal answer' + ds[0]['answer'])
print(ds[0]['scorer']('asdasd: '+ ds[0]['answer']))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
Sort these numbers in ascending order: -19.0, -10.0, 9.0, 4.0
Please follow the instruction below:
## 1. Let all your answers be a list of numbers. Instead of reporting your answer as -69, -13, 1, 7, 11, 43, 59, 61, use ['-69', '-13', '1', '7', '11', '43', '59', '61'] instead
## 2. Convert all numbers in the square brackets as strings. For example, ['-69', '-13', '1', '7', '11', '43', '59', '61']
Provide the final answer on last line after brainstorming.<|im_end|>
<|im_start|>assistant

larelappa
Final answer['-19.0', '-10.0', '4.0', '9.0']
['-19.0', '-10.0', '4.0', '9.0']
1.0


In [2]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial

# generate a number_sorting dataset

def needle_haystack_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    # name = last_line.strip().split()[-1].lower()
    name = last_line.lower().strip().split()
    print('Entry:', entry)
    if entry['answer'].lower().strip() in name:
        return 1.0
    return 0
    # return score_fn(name, entry)


def needle_haystack(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="needle_haystack",   # task name
        min_num_statements = 2,
        max_num_statements = 100,
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("needle_haystack")

    print(dataset[0])

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question']}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(needle_haystack_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = needle_haystack(tokenizer=tokenizer)
idx = 102
print(ds[idx]['prompt'])
print('larelappa\n' + ds[idx]['answer'], '*')
ans = ds[idx]['scorer']('Something\nFinal Answer: ' + str(ds[idx]['answer']))
# print(ds[idx]['scorer'](ans))

{'question': 'Caolain is neutral toward music. Alexx desires writing novels. Jake bears boxing. Harold gripes about dusting the furniture. Frederick disdains ironing the curtains. Cooper enjoys astronomy hobby. Caiden-Paul applauds all-terrain vehicles. Shayne delights in politics. Bradyn accepts artificial intelligence. Tyrnan supports climbing. Michal yearns for acting. Alvin deifies penguins. Allen relishes sailing. Brooke overlooks archery. Flynn prizes cleaning the patio. Grady can’t bear brewing beer. Rio ridicules acting. Wen is committed to emptying the dishwasher. Alfy execrates weeding the garden. Sweyn deifies bats. Emlyn laments bats. Shayan is passionate about snowboarding. Mehmet idolizes bird photography. Francis pines octopuses. Nikash worships ice skating. Tymom fancies motorcycles. Jaosha rejects balance. Abdur celebrates anime. Darryn bemoans logic. Michee revels in cleaning the ceiling fan. Khaleel worships trains. Jamie rails against the color amber. Daragh exults 

In [43]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial

# generate a number_sorting dataset

def syllogism(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="syllogism",   # task name
        allow_all = True,
        allow_no = True,
        allow_some = True,
        allow_some_not = True,
        invalid_ratio = 0.3,
        inversion_probability = 0.3,
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("syllogism")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        name = last_line.strip().split()[-1].capitalize()
        return score_fn(name, entry)

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = syllogism(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print('larelappa\n' + ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
Consider these statements:
1. Some birds are mortals
2. All mortals are spiders

Does it logically follow that:
Some birds are spiders?
(Answer Yes or No)Provide the final answer on last line after brainstorming.<|im_end|>
<|im_start|>assistant

larelappa
Yes
1.0


In [49]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def alice_in_wonderland(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="aiw",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("aiw")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        digits = re.findall(r'\d+', last_line)
        if digits:
            return score_fn(digits[0], entry)
        return 0

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = alice_in_wonderland(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print('larelappa\n' + ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
Jessica has 2 male friends and she also has 2 female friends. They all are friends with each other and have no other friends aside. How many female friends does Richard, a male friend of Jessica, have?Provide the final answer on last line after brainstorming.<|im_end|>
<|im_start|>assistant

larelappa
3
1.0


In [52]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def propositional_logic(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="propositional_logic",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("propositional_logic")

    def parser(llm_gen, entry):
        last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
        digits = re.findall(r'\d+', last_line)
        if digits:
            return score_fn(digits[0], entry)
        return 0

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(parser, entry=data)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = propositional_logic(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
The following question is a propositional logic reasoning question.

In the question we provide a list of premises. The task is to infer a correct conclusion from the premise.

FORMAT INSTRUCTIONS:
- Return the conclusion logic statement, as your final answer.
- Use the following notation to denote symbols
    - OR = ∨
    - AND = ∧
    - IMPLIES = →
    - IFF = ↔
    - NOT = ¬

Here is the question:
Given:
1. (P → P)
.2. ¬¬Q
.What can we conclude from the above statements?Provide the final answer on last line after brainstorming.<|im_end|>
<|im_start|>assistant

None


TypeError: can only concatenate str (not "NoneType") to str

In [ ]:
family_relationships
gsm_symbolic
list_functions

In [22]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def family_relationships_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))
    if len(last_line) == 0: return 0
    last_line = last_line[-1].strip().lower().split()
    answer = entry['answer'].lower().strip()
    if answer in last_line:
        p = last_line.index(answer)
        return 1 / len(last_line[p:])
    return 0

def family_relationships(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="family_relationships",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("family_relationships")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(family_relationships_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = family_relationships(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
James is married to Sophie. They have a child called Noah. Noah is married to Karen. They have children called Jayden and Hannah.

What relation is James to Jayden? Answer with a single word.Provide the final answer on last line after brainstorming.<|im_end|>
<|im_start|>assistant

grandfather
1.0


In [1]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def gsm_symbolic_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    digits = re.findall(r'\d+', last_line)
    if digits:
        return score_fn(digits[0], entry)
    return 0


def gsm_symbolic(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="gsm_symbolic",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("gsm_symbolic")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(gsm_symbolic_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = gsm_symbolic(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

/Users/ohi/Documents/GitHub/NanoAgent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
Robert is 20 years old. In 4 years his cousin Mary will be 3 times as old as Robert is now. How old is Mary right now? Give the result as your final answer. Do not include units.Provide the final answer on last line after brainstorming.<|im_end|>
<|im_start|>assistant

56
1.0


In [64]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def list_functions_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    last_line = last_line[last_line.find('['):]
    print('Fetched:', last_line[last_line.find('['):])
    return score_fn(last_line, entry)


def list_functions(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="list_functions",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("list_functions")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'}],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(list_functions_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = list_functions(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + ds[-1]['answer']))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
You are an expert at inductive reasoning. Generate an output corresponding to the given input.
The output is generated by applying the same rule that maps input to output for the examples provided. Your answer should be a list of element/elements
Examples:
Input 1: [1, 60, 49, 19, 23, 70, 45, 41]
Output 1: [41, 45, 70, 23, 19, 49, 60, 1]
Input 2: [98, 12, 8, 75, 94, 24, 34, 2, 39, 80]
Output 2: [80, 39, 2, 34, 24, 94, 75, 8, 12, 98]
Input 3: [36, 84, 93, 52]
Output 3: [52, 93, 84, 36]
Input 4: [73, 76, 92, 60, 47, 17]
Output 4: [17, 47, 60, 92, 76, 73]


Input: [15, 58, 83, 81, 20, 58, 39]
Output:
Provide the final answer on last line after brainstorming.<|im_end|>
<|im_start|>assistant

[39, 58, 20, 81, 83, 58, 15]
Fetched: [39, 58, 20, 81, 83, 58, 15]
1.0


In [12]:
def chain_sum_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    digits = re.findall(r'[+-]?\d+', last_line)
    if digits:
        digit = float(digits[-1])
        ans = float(entry['answer'])
        if digit == ans: return 1
        return abs(digit - ans) / 5
    return 0

def chain_sum(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="chain_sum",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("chain_sum")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [
                    {'role': 'user', 'content': data['question']},
                ],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(chain_sum_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list

import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = chain_sum(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
print(ds[-1]['scorer']('Something\nFinal Answer: ' + '-2'))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
State the final answer to the following arithmetic problem: 7 - 6 - 2 - 2 =<|im_end|>
<|im_start|>assistant

-3
0.2


In [13]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re

# generate a number_sorting dataset

def codeio_parser(llm_gen, entry, score_fn):
    # last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip()
    # llm_gen = llm_gen.strip().replace("```", "").replace('```json', '').strip()
    # print(llm_gen)
    if '{' in llm_gen and '}' in llm_gen:
        l = llm_gen.find('{')
        r = llm_gen.find('}')
        return score_fn(llm_gen[l:r+1], entry)
    return score_fn(llm_gen[l:r+1], entry)


def codeio(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="codeio",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("codeio")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [{'role': 'user', 'content': data['question'] + 'Provide the final answer on last line after brainstorming.'},
                 {'role': 'assistant', 'content': '```json\n'}],
                # add_generation_prompt=True,
                tokenize=False,
                continue_final_message=True,
            ),
            'answer': data['answer'],
            'scorer': partial(codeio_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list


import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = codeio(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
ans = 'asdasdasdasdasdasdasdasdasdasdSomething\nFinal Answer: ' + "```json\n" + ds[-1]['answer'] + '\n```'
print(ans)
print(ds[-1]['scorer'](ans))

<string>:27: SyntaxWarning: invalid escape sequence '\d'
<string>:30: SyntaxWarning: invalid escape sequence '\d'
<string>:40: SyntaxWarning: invalid escape sequence '\d'
<string>:7: SyntaxWarning: invalid escape sequence '\s'
<string>:72: SyntaxWarning: invalid escape sequence '\('
<string>:79: SyntaxWarning: invalid escape sequence '\s'
<string>:85: SyntaxWarning: invalid escape sequence '\s'
<string>:89: SyntaxWarning: invalid escape sequence '\s'
<string>:95: SyntaxWarning: invalid escape sequence '\s'
<string>:7: SyntaxWarning: invalid escape sequence '\p'
<string>:42: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
<string>:56: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
<string>:19: SyntaxWarning: invalid escape sequence '\l'
<string>:27: SyntaxWarning: invalid escape sequence '\l'


Error(pQueue.insert): Invalid insert operation. Queue is full
<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user

You are given a question that requires some input and output variables as follows:

In a game of Pig, where two players take turns rolling a die, the goal is to be the first to reach 40 points. On each turn, a player can either roll the die or hold. If a player rolls a 1, they score nothing and it becomes the opponent's turn. If a player rolls any other number, the number is added to their pending points. A player can choose to hold to add their pending points to their score and end their turn. Given the current state of the game, which includes the current player's turn, the scores of both players, and the pending points, what is the optimal action for the current player to take?

The input and output requirements are as follows:

Input:
  `player` (int): The current player's turn (0 or 1).
  `me` (int): The current score of the player whose tur

In [ ]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re


def acre_parser(llm_gen, entry, score_fn):
    last_line = list(filter(lambda x: len(x.strip()) > 0, llm_gen.split('\n')))[-1].strip().lower()
    ans = entry['answer'].strip().lower()
    if ans in last_line.split():
        score = score_fn(last_line, entry) * 0.5
        judge_score = 0 #response_judge(entry['question'], response=llm_gen, n_tokens=JUDGE_TOKENS, ref_answer=entry['answer'])[1]
        return score + judge_score * 0.5
    return 0


def acre(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="acre",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("acre")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [
                    {'role': 'user', 'content': data['question']},
                ],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(acre_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list

import json
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = acre(tokenizer=tokenizer)

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
ans = 'asdasdasdasdasdasdasdasdasdasdSomething\nFinal Answer: "' + ds[-1]['answer'] + 'on."'
print(ans)
print(ds[-1]['scorer'](ans))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
You are a researcher studying causal relationships using Blicket experiments. In these experiments, certain objects (called 'blickets') have the hidden property of activating a detector, causing its light to turn on.

Each example shows the results of placing different combinations of objects on the detector. Each object is described by color, material and shape. Your task is to determine whether a new combination of objects will cause the detector to activate.

After observing the previous examples, respond with:
- "on" if you can determine the detector light will turn on
- "off" if you can determine the detector light will stay off
- "undetermined" if there is insufficient evidence to reach a conclusion

Do not use quotation marks in your answer.

Previous experimental results:
cyan rubber cylinder, gray metal cube → on
gray metal cube → off
cyan rubber cylinder → on
gray metal cylinder, green rubber cylin

In [18]:
import reasoning_gym
from reasoning_gym import get_score_answer_fn
from functools import partial
import re
import string

def last_line_parser(inp):
    last_line = list(filter(lambda x: len(x.strip().strip(string.punctuation).strip()) > 0, inp.split('\n')))
    if last_line:
        last_line = last_line[-1].strip().lower()
        return last_line
    return ''

def word_parser(inp):
    inp = inp.lower().split()
    words = list(map(lambda x: x.strip(string.punctuation), inp))
    words = list(filter(lambda x: x.strip() != '', words))
    return words

def zebra_puzzles_parser(llm_gen, entry, score_fn):
    last_line = last_line_parser(llm_gen)
    words = word_parser(last_line)
    ans = entry['answer'].strip().lower()
    print(ans, '->', words)
    score = 0
    if ans in words:
        p = words.index(ans)
        score = (1 / len(words[p:])) * 0.5
    judge_score = 0 #response_judge(entry['question'], response=llm_gen, n_tokens=JUDGE_TOKENS, ref_answer=entry['answer'])[1]
    return min(judge_score + score, 1)


def zebra_puzzles(tokenizer, size=500, prompt_token_len=None):
    dataset = reasoning_gym.create_dataset(
        name="zebra_puzzles",   # task name
        seed = 42,
        size = size,
    )
    score_fn = get_score_answer_fn("zebra_puzzles")

    dataset_list = []
    for data in dataset:
        dataset_list.append({
            'prompt': tokenizer.apply_chat_template(
                [
                    {'role': 'user', 'content': data['question']},
                ],
                add_generation_prompt=True,
                tokenize=False,
                continue_final_message=False,
            ),
            'answer': data['answer'],
            'scorer': partial(zebra_puzzles_parser, entry=data, score_fn=score_fn)
        })

    if prompt_token_len:
        dataset_list = list(filter(lambda x: len(tokenizer.encode(x['prompt'])) <= prompt_token_len, dataset_list))

    return dataset_list

import json
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("quwsarohi/NanoAgent-135M")
ds = zebra_puzzles(tokenizer=tokenizer)

phrase = '''1. Alice is directly left of the person partial to Pall Mall.
2. The person partial to Pall Mall and Bob are next to each other.
3. The person partial to Pall Mall is the person who loves the boquet of iris.
4. The person's child is named Bella is Bob.
5. The person who is the mother of Billy is the person who loves the boquet of lilies.
6. The person who loves the boquet of lilies is the Prince smoker.
7. The person who loves a carnations arrangement is the person who smokes Blue Master.
8. Arnold is the person partial to Pall Mall.
9. The person's child is named Alice is Alice.
10. The person who smokes Blue Master is in the second house.'''

print(ds[-1]['prompt'])
print(ds[-1]['answer'])
ans = 'asdasdasdasdasdasdasdasdasdasdSomething\nFinal Answer: "' + phrase
print(ans)
print(ds[-1]['scorer'](ans))

<|im_start|>system
You are a helpful AI assistant. <|im_end|>
<|im_start|>user
This is a logic puzzle. There are 4 houses (numbered 1 on the left, 4 on the right), from the perspective of someone standing across the street from them. Each has a different person in them. They have different characteristics:
 - Each person has a unique name: carol, arnold, alice, bob
 - Each mother is accompanied by their child: alice, timothy, billy, bella
 - They all have a different favorite flower: iris, daffodils, carnations, lilies
 - Everyone has a different favorite cigar: blue master, pall mall, prince, dunhill

1. Alice is directly left of the person partial to Pall Mall.
2. The person partial to Pall Mall and Bob are next to each other.
3. The person partial to Pall Mall is the person who loves the boquet of iris.
4. The person's child is named Bella is Bob.
5. The person who is the mother of Billy is the person who loves the boquet of lilies.
6. The person who loves the boquet of lilies is th